# 08 · Results, figures and model index  *(branch `exp/08-results`)*

Collects everything for the thesis (brief §4): one `results.csv` (every run, every seed, every metric),
summary tables in Markdown, the required figures, and an index from every reported number to its checkpoint.
Nothing is trained here.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/08-results"                                  # the branch this notebook belongs to
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
import json, shutil, pandas as pd, matplotlib.pyplot as plt
from rltrade import results as R
df = R.load_results(L.runs)
print(len(df), "runs |", df.groupby("experiment").size().to_dict())
shutil.copy(L.runs / "results.csv", L.report / "results.csv")

## Summary tables (mean ± std over seeds, test period, net of costs)

In [ ]:
parts = ["# Results (test period, net of costs; mean ± std over seeds)\n",
         "Costs: fee {:.2%} + slippage {:.2%} per side; decisions every {} min; fills at next bar.\n".format(
             df.fee_rate.iloc[0], df.slippage.iloc[0], int(df.step_gap.iloc[0]))]
for exp, sub in df.groupby("experiment"):
    parts += [f"\n## {exp}\n", R.markdown_table(R.summarise(sub, metrics=["val_sharpe"] + R.HEADLINE),
                                                   metrics=["val_sharpe"] + R.HEADLINE), ""]
text = "\n".join(parts); (L.report / "results_tables.md").write_text(text, encoding="utf-8"); print(text)

## Final model — chosen on validation across all reported configurations

In [ ]:
MIN_TRADES = 5   # only configurations whose checkpoints were chosen under the current eligibility rule
cand = df[~df.agent.isin(["buy_and_hold", "random", "cash"]) & (df.signal_source == "lagged")]
cand = cand[pd.to_numeric(cand.select_min_trades, errors="coerce") >= MIN_TRADES]
g = cand.groupby(R.GROUP, dropna=False)
best_cfg = g.val_sharpe.mean().idxmax()
runs = cand.set_index(R.GROUP).loc[best_cfg].reset_index()
print("configuration:", dict(zip(R.GROUP, best_cfg)))
print(runs[["run_id", "seed", "val_sharpe", "test_sharpe", "test_cum_return_pct", "test_max_drawdown_pct"]])

## Figures

In [ ]:
bh = df[df.agent == "buy_and_hold"].drop_duplicates("experiment").head(1)
A6 = "06_agents_mt5" if (df.experiment == "06_agents_mt5").any() else "06_agents"
if (df.experiment == A6).any():
    a = df[df.experiment == A6]
    groups = {x: a[a.agent == x] for x in a.agent.unique() if x not in ("buy_and_hold", "cash")}; groups["Buy & hold"] = bh; groups["Cash"] = a[a.agent == "cash"]
    R.plot_curves(df, L.runs, groups, path=L.report / "fig_equity_all_agents.png"); plt.show()
    R.plot_curves(df, L.runs, groups, kind="drawdown", path=L.report / "fig_drawdown_all_agents.png"); plt.show()
AB = sorted(e for e in df.experiment.unique() if str(e).startswith("05_ablation_"))   # tuned re-run of 05
if AB:      # its state D is the same agent's 06 run (see notebook 05)
    ab_agent = df[(df.experiment == AB[-1]) & ~df.agent.isin(["buy_and_hold", "cash"])].agent.iloc[0]
    ab = df[((df.experiment == AB[-1]) | ((df.experiment == A6) & (df.state == "D")))
            & (df.agent == ab_agent) & (df.signal_source == "lagged")]
    R.plot_ablation(R.summarise(ab), L.report / "fig_ablation.png",
                    title=f"Ablation over the state vector, {ab_agent.upper()} (test, mean ± std over seeds)"); plt.show()
elif (df.experiment == "05_ablation").any():
    ab = df[(df.experiment == "05_ablation") & (df.agent != "buy_and_hold") & (df.signal_source == "lagged")]
    R.plot_ablation(R.summarise(ab), L.report / "fig_ablation.png"); plt.show()
finals = L.rnn_tuning / "final_runs.csv"
if finals.exists():
    fr = pd.read_csv(finals)
    R.plot_rnn_history({f"{r.arch} s{r.seed}": r.dir for r in fr.itertuples() if r.seed == fr.seed.min()},
                       L.report / "fig_rnn_loss.png"); plt.show()
elif (L.rnn_default / "history.json").exists():
    R.plot_rnn_history({"enhanced (default)": L.rnn_default}, L.report / "fig_rnn_loss.png"); plt.show()
print("Optuna figures already written by 04/06:", sorted(p.name for p in L.report.glob("*optuna*")))

## Model index — every reported number → one checkpoint

In [ ]:
idx = df[["run_id", "experiment", "agent", "state", "signal_source", "reward", "seed", "commit", "test_sharpe"]].copy()
idx["folder"] = [str(L.runs / "models" / r) for r in idx.run_id]
idx.to_csv(L.report / "model_index.csv", index=False); print(idx.head(20))